In [1]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, mean_squared_error
import pandas as pd

In [2]:
def real_formatter(x, pos):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

In [3]:
# Carregando o Dataset pré processado
df_limpo = pd.read_csv('Housing_pre_processing.csv')

In [4]:
# Dados de treino e teste
X = df_limpo.drop(columns=['preco'])
y = df_limpo['preco']

In [5]:
# Dividindo os dados
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.20, random_state=42)

In [6]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42) # Criando o Kfold

In [7]:
regressor_target_mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(),
    transformer=StandardScaler()
)

In [8]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', regressor_target_mlp)
])

In [9]:
# Ajuste fino do algoritmo de rede neural
params_mlp = {
    'mlp__regressor__hidden_layer_sizes': [
        (128,)
    ],
    
    'mlp__regressor__activation': [
        'relu'
    ],
    
    'mlp__regressor__solver': [
        'adam'
    ],
    
    'mlp__regressor__alpha': [
        1e-4,
        1e-3,
        1e-2
    ],
    
    'mlp__regressor__learning_rate_init': [
        0.0001,
        0.001,
    ],
    
    'mlp__regressor__max_iter': [
        1000,
        2000
    ],
    
    'mlp__regressor__early_stopping': [
        True
    ],
    
    'mlp__regressor__random_state': [
        42
    ]
}


In [10]:
grid_search_mlp = GridSearchCV(
    pipeline,
    params_mlp,
    cv=kfold,
    n_jobs=-1,
    scoring='r2',
    verbose=1
)

In [11]:
grid_search_mlp.fit(X_treino, y_treino)

Fitting 10 folds for each of 12 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...rdScaler()))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'mlp__regressor__activation': ['relu'], 'mlp__regressor__alpha': [0.0001, 0.001, ...], 'mlp__regressor__early_stopping': [True], 'mlp__regressor__hidden_layer_sizes': [(128,)], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosit

In [12]:
print(grid_search_mlp.best_params_)
print("-"*100)
print(grid_search_mlp.best_score_)

{'mlp__regressor__activation': 'relu', 'mlp__regressor__alpha': 0.01, 'mlp__regressor__early_stopping': True, 'mlp__regressor__hidden_layer_sizes': (128,), 'mlp__regressor__learning_rate_init': 0.001, 'mlp__regressor__max_iter': 1000, 'mlp__regressor__random_state': 42, 'mlp__regressor__solver': 'adam'}
----------------------------------------------------------------------------------------------------
0.6581932530559768


In [13]:
y_pred = grid_search_mlp.predict(X_teste)

In [14]:
r2_score = r2_score(y_teste, y_pred)

In [15]:
print(f"Coeficiente de determinação: {r2_score * 100:.2f}%")

Coeficiente de determinação: 66.09%


In [16]:
# Métricas 
mae_mlp = mean_absolute_error(y_teste, y_pred)
rmse_mlp = root_mean_squared_error(y_teste, y_pred)

print(f'MAE: {mae_mlp:.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f'RMSE: {rmse_mlp:.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'))

MAE: 950021,63
RMSE: 1288570,61
